In [1]:
# ── Cell 0: Install dependencies ─────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-U", "bitsandbytes>=0.46.1"], check=True)
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.8 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
Dependencies installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

In [2]:
# ── Cell 1: Auth ──────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN2"))
print("Authenticated")

# ── Cell 2: Imports ───────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch
import pandas as pd
from pathlib import Path

# ── Cell 3: Shared config ─────────────────────────────────────────────────────
MODEL_ID    = "meta-llama/Meta-Llama-3-8B-Instruct"
ADAPTER_DIR = "/kaggle/input/datasets/imanghotbi/lora-finetuned/llama3-bias-lora-v5"
DATA_PATH   = "/kaggle/input/datasets/imanghotbi/allsides3/allsides_clean_splits.csv"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

def build_prompt(article_text):
    return f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a neutral news summarizer. Your task is to generate a balanced, \
factual summary of the provided news article without introducing bias or \
favoring any political perspective.<|eot_id|><|start_header_id|>user<|end_header_id|>
ARTICLE:
{article_text}

Write a neutral, factual summary (2-4 sentences) of this article.<|eot_id|>\
<|start_header_id|>assistant<|end_header_id|>
"""

def generate_summary(article_text, model, tokenizer, max_new_tokens=150):
    prompt = build_prompt(article_text)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def run_inference(test_df, model, tokenizer, output_path):
    results = []
    for i, (_, row) in enumerate(test_df.iterrows()):
        for stance in ["left", "center", "right"]:
            article_text = str(row[f"{stance}_text"])
            summary = generate_summary(article_text, model, tokenizer)
            results.append({
                "example_id":        i,
                "issue":             row["issue"],
                "topic":             row["topic"],
                "stance":            stance,
                "input_article":     article_text,
                "generated_summary": summary,
                "roundup_text":      row["roundup_text"],
            })
        if (i + 1) % 10 == 0:
            pd.DataFrame(results).to_csv(output_path, index=False)
            print(f"Progress: {i+1}/{len(test_df)}")
    pd.DataFrame(results).to_csv(output_path, index=False)
    print(f"Done. Total rows: {len(results)}")

# ── Cell 4: Load clean test set ───────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print(f"Clean test examples: {len(test_df)}")
print(f"Expected: 307 rows")


Authenticated
Clean test examples: 307
Expected: 307 rows


In [ ]:
# ── Cell 5: ZERO-SHOT inference ───────────────────────────────────────────────
print("Loading base model for zero-shot...")
tokenizer_zs = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer_zs.pad_token = tokenizer_zs.eos_token

model_zs = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model_zs.eval()
print(f"Zero-shot model loaded — {next(model_zs.parameters()).device}")

run_inference(
    test_df, model_zs, tokenizer_zs,
    "/kaggle/working/zeroshot_clean_summaries.csv"
)

# ── Cell 6: Free memory ───────────────────────────────────────────────────────
import gc
gc.collect()
torch.cuda.empty_cache()
print("Memory freed")

# ── Cell 7: FINE-TUNED inference ──────────────────────────────────────────────


print("Loading fine-tuned model...")
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_ID)  # load from base model
tokenizer_ft.pad_token = tokenizer_ft.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model_ft = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_ft.eval()
print(f"Fine-tuned model loaded — {next(model_ft.parameters()).device}")

run_inference(
    test_df, model_ft, tokenizer_ft,
    "/kaggle/working/finetuned_clean_summaries.csv"
)

Memory freed
Loading fine-tuned model...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Fine-tuned model loaded — cuda:0
Progress: 10/307
Progress: 20/307
Progress: 30/307
Progress: 40/307
Progress: 50/307
Progress: 60/307
Progress: 70/307
Progress: 80/307
Progress: 90/307
Progress: 100/307
Progress: 110/307
Progress: 120/307
Progress: 130/307
Progress: 140/307
Progress: 150/307
Progress: 160/307
Progress: 170/307
Progress: 180/307
Progress: 190/307
Progress: 200/307
Progress: 210/307
Progress: 220/307
Progress: 230/307
Progress: 240/307
Progress: 250/307
Progress: 260/307
Progress: 270/307
Progress: 280/307
Progress: 290/307
Progress: 300/307
Done. Total rows: 921
